In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [51]:
data=pd.read_csv("D:\KaggleCompetitions\Email_Spam\email.csv")
data.head(3)

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


In [52]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5573 entries, 0 to 5572
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5573 non-null   object
 1   Message   5573 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [53]:
spam_cat=data[['Category']]

In [54]:
Mail=data['Message'].astype(str)
target=data['Category']

In [55]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(Mail,target,test_size=0.2,random_state=42)

In [56]:
y_test

3690     ham
3527     ham
724      ham
3370     ham
468      ham
        ... 
2942     ham
5464     ham
3227     ham
3795     ham
2879    spam
Name: Category, Length: 1115, dtype: object

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline

pipeline=Pipeline([
    ('tfidf',TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english',
        max_df=0.9
    )),
    ('sgd',SGDClassifier(loss='log_loss',random_state=42))
])

pipeline.fit(x_train,y_train)

,steps,"[('tfidf', ...), ('sgd', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [58]:
y_prob=pipeline.predict_proba(x_test)
y_pred=(y_prob[:1]>0.45).astype(int)

print(y_pred) 
# y_prob

[[1 0 0]]


In [59]:
y_pred.shape

(1, 3)

## Tuning : Character n-grams 

In [60]:
pipeline_char = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char',
        ngram_range=(3,5),
        min_df=3
    )),
    ('sgd', SGDClassifier(loss='log_loss', random_state=42))
])

pipeline_char.fit(x_train, y_train)
y_pred_char = pipeline_char.predict(x_test)


In [61]:
y_pred=pipeline.predict(x_test)

In [62]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_char))
print(classification_report(y_test, y_pred_char))

[[958   0]
 [ 24 133]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       958
        spam       1.00      0.85      0.92       157

    accuracy                           0.98      1115
   macro avg       0.99      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



## Tuning 3 : Class weighting 

In [63]:
pipeline_weighted = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english',
        max_df=0.9
    )),
    ('sgd', SGDClassifier(
        loss='log_loss',
        class_weight='balanced',
        random_state=42
    ))
])

pipeline_weighted.fit(x_train, y_train)
y_pred_weighted = pipeline_weighted.predict(x_test)


In [64]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_weighted))
print(classification_report(y_test, y_pred_weighted))

[[952   6]
 [ 12 145]]
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       958
        spam       0.96      0.92      0.94       157

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.97      1115
weighted avg       0.98      0.98      0.98      1115



In [65]:
pipeline_alpha = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1,2),
        stop_words='english',
        max_df=0.9
    )),
    ('sgd', SGDClassifier(
        loss='log_loss',
        alpha=1e-4,   # try 1e-3, 1e-5 later
        random_state=42
    ))
])

pipeline_alpha.fit(x_train, y_train)
y_pred_alpha = pipeline_alpha.predict(x_test)


In [66]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_alpha))
print(classification_report(y_test, y_pred_alpha))

[[956   2]
 [ 21 136]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       958
        spam       0.99      0.87      0.92       157

    accuracy                           0.98      1115
   macro avg       0.98      0.93      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [67]:
print(y_train.value_counts())

Category
ham               3867
spam               590
{"mode":"full"       1
Name: count, dtype: int64


In [68]:
print(x_train[y_train == '{"mode":"full"'])

5572    isActive:false}
Name: Message, dtype: object


In [69]:
from sklearn.model_selection import GridSearchCV

mask=y_train.isin(['ham','spam'])
x_train=x_train[mask]
y_train=y_train[mask]

param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2)],
    'tfidf__max_features': [1000,3000,5000],
    'tfidf__stop_words': ['english',None],
    'tfidf__max_df': [0.7, 0.8, 0.9],
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

grid.fit(x_train, y_train) 

best_model = grid.best_estimator_ 
y_pred_grid = best_model.predict(x_test)

print(grid.best_params_)
print(grid.best_score_)

{'tfidf__max_df': 0.7, 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 2), 'tfidf__stop_words': None}
0.9576347258708291


In [70]:
data.shape

(5573, 2)

# Another dataset

In [3]:

data1=pd.read_csv(r"D:\Complete_proj\Email_spam\model\data\enron_spam_data.csv")

In [4]:
data1.shape


(33716, 5)

In [5]:
# data1.head(5)
spam_data=data1.copy()
spam_data.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [6]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  33716 non-null  int64 
 1   Subject     33716 non-null  object
 2   Message     33664 non-null  object
 3   Spam/Ham    33716 non-null  object
 4   Date        33716 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [7]:
spam_data.drop(columns=['Unnamed: 0','Subject','Date'], inplace=True)

In [8]:
spam_data.head()

,Message,Spam/Ham
0,NaN,ham
1,"gary , production from the high island larger ...",ham
2,- calpine daily gas nomination 1 . doc,ham
3,fyi - see note below - already done .\nstella\...,ham
4,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham


In [9]:
spam_data.shape

(33716, 2)

In [15]:
spam_data.rename(columns={'Spam/Ham':'target'},inplace=True)
spam_data.head()

,Message,target
0,NaN,ham
1,"gary , production from the high island larger ...",ham
2,- calpine daily gas nomination 1 . doc,ham
3,fyi - see note below - already done .\nstella\...,ham
4,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham


In [16]:
spam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33716 entries, 0 to 33715
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Message  33664 non-null  object
 1   target   33716 non-null  object
dtypes: object(2)
memory usage: 526.9+ KB


In [17]:
spam_data['target'].value_counts()

target
spam    17171
ham     16545
Name: count, dtype: int64

In [18]:
from sklearn.preprocessing import LabelEncoder
lbl_encoder=LabelEncoder()

In [19]:
spam_data['target']=lbl_encoder.fit_transform(spam_data['target'])

In [20]:
spam_data.head()

,Message,target
0,NaN,0
1,"gary , production from the high island larger ...",0
2,- calpine daily gas nomination 1 . doc,0
3,fyi - see note below - already done .\nstella\...,0
4,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0


In [22]:
spam_data['target'].value_counts()

target
1    17171
0    16545
Name: count, dtype: int64

In [21]:
spam_data.isnull().sum()

Message    52
target      0
dtype: int64

In [29]:
spam_data.duplicated(subset=['Message']).sum()

np.int64(17922)

In [30]:
# View the messages that are being flagged as duplicates
duplicates_only = spam_data[spam_data.duplicated(subset=['Message'], keep=False)]
print(duplicates_only.sort_values(by='Message').head(10))

                                                 Message  target
12000  ' for the record ' questions for the enside ne...       0
12002  ' for the record ' questions for the enside ne...       0
2137   ( see attached file : hplnl 201 . xls )\n- hpl...       0
2139   ( see attached file : hplnl 201 . xls )\n- hpl...       0
2686   ( see attached file : hplno 310 . xls )\n- hpl...       0
2687   ( see attached file : hplno 310 . xls )\n- hpl...       0
2703   ( see attached file : hplno 313 . xls )\n- hpl...       0
2701   ( see attached file : hplno 313 . xls )\n- hpl...       0
2710   ( see attached file : hplno 314 . xls )\n- hpl...       0
2708   ( see attached file : hplno 314 . xls )\n- hpl...       0


In [31]:
# This keeps the first occurrence and removes the rest
spam_data = spam_data.drop_duplicates(subset=['Message'], keep='first')

# Check the new distribution
print(spam_data['target'].value_counts())

target
0    15794
Name: count, dtype: int64


In [24]:
spam_data[spam_data.duplicated(subset=['Message'], keep=False)]

,Message,target
0,NaN,0
2,- calpine daily gas nomination 1 . doc,0
64,teco tap 60 . 000 / enron,0
148,"recognizing enron  , s increasing worldwide p...",0
183,NaN,0
...,...,...
33711,"fyi , kim .\n- - - - - original message - - - ...",1
33712,"fyi , kim .\n- - - - - original message - - - ...",1
33713,"fyi , kim .\n- - - - - original message - - - ...",1
33714,"fyi , kim .\n- - - - - original message - - - ...",1


In [98]:
#dropping duplicate values 
spam_data.drop_duplicates(inplace=True)

In [ ]:
spam_data.shape 

(15800, 2)

In [ ]:
#removing null rows as they are very less 
spam_data.dropna(subset=['Message'], inplace=True) 

In [ ]:
spam_data.isnull().sum() 

Message    0
target     0
dtype: int64

In [26]:
spam_data['target'].value_counts()

target
1    17171
0    16545
Name: count, dtype: int64

## EDA on data